In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install neuralforecast -q
!pip install kaleido==0.2.1 -q
!pip install workalendar -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 78.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.
   ━━━━━━

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from workalendar.europe import Russia
from tqdm import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
HORIZON_NAME = "1m"

In [ ]:
df = pd.read_csv("df_final+whether.csv", parse_dates=["timestamp"])
houses = [col for col in df.columns if col.startswith("house_")]

cal = Russia()
def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df["is_holiday"] = df["timestamp"].apply(is_holiday)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "MAPE": round(mape, 3)}

horizons = {
    "4h":  8, "8h": 16, "24h": 48,
    "7d":  336, "14d": 672, "1m": 1488,
}

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

dfs = []
for house in houses:
    tmp = pd.DataFrame({
        "unique_id": house,
        "ds": df["timestamp"],
        "y": df[house],
    })
    dfs.append(tmp)

df_nf = pd.concat(dfs, ignore_index=True)

df_val = df_nf[
    (df_nf["ds"] >= df["timestamp"].iloc[train_end]) &
    (df_nf["ds"] < df["timestamp"].iloc[val_end])
]
df_test = df_nf[df_nf["ds"] >= df["timestamp"].iloc[val_end]]

val_size = len(df_val) // len(houses)
test_size = len(df_test) // len(houses)

horizon_points = horizons[HORIZON_NAME]

In [ ]:
torch.cuda.empty_cache()
gc.collect()

nf = NeuralForecast(
    models=[
        TFT(
            h=horizon_points,
            input_size=48,
            hidden_size=32,
            n_head=2,
            max_steps=200,
            val_check_steps=50,
            early_stop_patience_steps=5,
            scaler_type="minmax",
            accelerator="gpu",
            devices=1,
            batch_size=1,
            windows_batch_size=1,
        )
    ],
    freq="30min"
)

cv_df = nf.cross_validation(
    df=df_nf,
    val_size=val_size,
    test_size=test_size,
    n_windows=None,
)

results = {}
for house in houses:
    metrics = evaluate(
        cv_df[cv_df["unique_id"] == house]["y"].values,
        cv_df[cv_df["unique_id"] == house]["TFT"].values
    )
    results[house] = metrics
    print(f"{house}: MAE={metrics['MAE']:.3f}, RMSE={metrics['RMSE']:.3f}, MAPE={metrics['MAPE']:.3f}%")

rows = [{"модель": "TFT", "дом": h, "горизонт": HORIZON_NAME, **results[h]} for h in houses]
pd.DataFrame(rows).to_csv(f"results_tft_{HORIZON_NAME}.csv", index=False)

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                    ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                    │ MAE                      │      0 │ train │     0 │
│ 1 │ padder_train            │ ConstantPad1d            │      0 │ train │     0 │
│ 2 │ scaler                  │ TemporalNorm             │      0 │ train │     0 │
│ 3 │ embedding               │ TFTEmbedding             │    128 │ train │     0 │
│ 4 │ temporal_encoder        │ TemporalCovariateEncoder │ 39.6 K │ train │     0 │
│ 5 │ temporal_fusion_decoder │ TemporalFusionDecoder    │ 17.0 K │ train │     0 │
│ 6 │ output_adapter          │ Linear                   │     33 │ train │     0 │
└───┴─────────────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 56.8 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 56.8 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 88                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

OutOfMemoryError: CUDA out of memory. Tried to allocate 4.01 GiB. GPU 0 has a total capacity of 39.49 GiB of which 699.44 MiB is free. Including non-PyTorch memory, this process has 38.80 GiB memory in use. Of the allocated memory 38.26 GiB is allocated by PyTorch, and 42.85 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
rows = [{"модель": "TFT", "дом": h, "горизонт": "1m",
         "MAE": None, "RMSE": None, "MAPE": None} for h in houses]
pd.DataFrame(rows).to_csv("results_tft_1m.csv", index=False)